In [6]:
from neo4j import GraphDatabase
import pandas as pd

# Thay thế bằng thông tin đăng nhập Neo4j của bạn
NEO4J_URI = "neo4j://127.0.0.1:7687"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = "toanphat1509"

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

# Kiểm tra kết nối
try:
    driver.verify_connectivity()
    print("Kết nối Neo4j thành công!")
except Exception as e:
    print(f"Lỗi kết nối Neo4j: {e}")

# Hàm để đóng driver khi không sử dụng nữa
def close_driver():
    driver.close()
    print("Đã đóng kết nối Neo4j.")

Kết nối Neo4j thành công!


In [7]:
movies_df = pd.read_csv('movies.csv')
ratings_df = pd.read_csv('ratings.csv')

print("Đã load dữ liệu phim:")
print(movies_df.head())

print("\nĐã load dữ liệu đánh giá:")
print(ratings_df.head())

Đã load dữ liệu phim:
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  

Đã load dữ liệu đánh giá:
   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931


In [8]:
def create_all_constraints(tx):
    tx.run("CREATE CONSTRAINT FOR (m:Movie) REQUIRE m.movieId IS UNIQUE")
    tx.run("CREATE CONSTRAINT FOR (u:User) REQUIRE u.userId IS UNIQUE")
    tx.run("CREATE CONSTRAINT FOR (g:Genre) REQUIRE g.name IS UNIQUE")
    print("Đã tạo ràng buộc cho Movie(movieId), User(userId) và Genre(name).")

with driver.session() as session:
    session.execute_write(create_all_constraints)

Đã tạo ràng buộc cho Movie(movieId), User(userId) và Genre(name).


In [9]:
def import_movies(tx, movie_data):
    query = """
    UNWIND $movie_data AS row
    MERGE (m:Movie {movieId: row.movieId})
    ON CREATE SET m.title = row.title, m.genres = row.genres
    """
    tx.run(query, movie_data=movie_data)

def import_ratings(tx, rating_data):
    query = """
    UNWIND $rating_data AS row
    MERGE (u:User {userId: row.userId})
    MERGE (m:Movie {movieId: row.movieId})
    WITH u, m, row
    WHERE row.rating >= 3  // Chỉ tạo mối quan hệ nếu rating >= 3
    CREATE (u)-[r:RATED {rating: row.rating, timestamp: row.timestamp}]->(m)
    """
    tx.run(query, rating_data=rating_data)

print("Đang import dữ liệu phim...")
with driver.session() as session:
    movie_records = movies_df.to_dict('records')
    batch_size = 1000
    for i in range(0, len(movie_records), batch_size):
        batch = movie_records[i:i + batch_size]
        session.execute_write(import_movies, batch)
print(f"Đã import {len(movies_df)} phim vào Neo4j.")

print("\nĐang import dữ liệu đánh giá và tạo mối quan hệ...")
with driver.session() as session:
    rating_records = ratings_df.to_dict('records')
    batch_size = 1000
    for i in range(0, len(rating_records), batch_size):
        batch = rating_records[i:i + batch_size]
        session.execute_write(import_ratings, batch)
print(f"Đã xử lý {len(ratings_df)} đánh giá và tạo mối quan hệ tương ứng trong Neo4j.")

Đang import dữ liệu phim...
Đã import 9742 phim vào Neo4j.

Đang import dữ liệu đánh giá và tạo mối quan hệ...
Đã xử lý 100836 đánh giá và tạo mối quan hệ tương ứng trong Neo4j.


In [10]:
all_genres = movies_df['genres'].str.split('|').explode().unique()
print(f"Tìm thấy {len(all_genres)} thể loại duy nhất.")

def create_genre_nodes_and_relationships(tx, movie_data_for_genres, genres):
    # Tạo các node Genre trước
    genre_query = """
    UNWIND $genres AS genre_name
    MERGE (g:Genre {name: genre_name})
    """
    tx.run(genre_query, genres=genres.tolist())

    # Tạo mối quan hệ HAS_GENRE
    rel_query = """
    UNWIND $movie_data AS row
    MATCH (m:Movie {movieId: row.movieId})
    UNWIND split(row.genres, '|') AS genre_name
    MATCH (g:Genre {name: genre_name})
    MERGE (m)-[:HAS_GENRE]->(g)
    """
    tx.run(rel_query, movie_data=movie_data_for_genres)

print("Đang tạo các node Genre và mối quan hệ HAS_GENRE...")
with driver.session() as session:
    movie_records_for_genres = movies_df[['movieId', 'genres']].to_dict('records')
    batch_size = 1000
    # Batch processing for relationships might be needed if movie_records_for_genres is very large
    # For this example, we'll pass all genres once and movie data in batches
    for i in range(0, len(movie_records_for_genres), batch_size):
        batch = movie_records_for_genres[i:i + batch_size]
        session.execute_write(create_genre_nodes_and_relationships, batch, all_genres)
print("Đã tạo các node Genre và mối quan hệ HAS_GENRE.")

Tìm thấy 20 thể loại duy nhất.
Đang tạo các node Genre và mối quan hệ HAS_GENRE...
Đã tạo các node Genre và mối quan hệ HAS_GENRE.


In [11]:
close_driver()

Đã đóng kết nối Neo4j.
